In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 75 — DSPy Classifier Optimizer

## Goal
Build a text classifier pipeline in **DSPy** and let
DSPy **optimize prompts/modules** automatically.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **DSPy Signatures** | Declarative input → output specifications |
| **Modules** | `Predict`, `ChainOfThought` — composable LM functions |
| **Examples** | `dspy.Example` for training/eval data |
| **Metrics** | Functions that score predictions |
| **Optimizers** | `LabeledFewShot` — automatic prompt compilation |

## Stack
- `dspy` 3.1+
- Ollama `qwen3.5:9b` via LiteLLM

In [ ]:
import os, warnings
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import dspy
import random

print(f"DSPy version: {dspy.__version__}")

## Step 1 — Configure DSPy with Ollama

DSPy uses LiteLLM under the hood. For Ollama,
use the `ollama_chat/model_name` format.

In [ ]:
# Configure DSPy to use local Ollama
lm = dspy.LM(
    "ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    api_key="fake",  # required by LiteLLM but not used by Ollama
    temperature=0,
)
dspy.configure(lm=lm)

# Quick test
response = lm("Say 'DSPy is working!' and nothing else.")
print(f"LM response: {response[0][:100]}")

## Step 2 — Create Classification Dataset

We'll classify customer support tickets into categories.
Each example has a `text` (input) and `category` (label).

In [ ]:
# Customer support ticket dataset
raw_data = [
    # Billing
    {"text": "I was charged twice for my subscription last month", "category": "billing"},
    {"text": "Can I get a refund for the unused portion of my plan?", "category": "billing"},
    {"text": "My credit card was declined but I have sufficient funds", "category": "billing"},
    {"text": "I need to update my payment method to a new card", "category": "billing"},
    {"text": "The annual billing amount doesn't match the quoted price", "category": "billing"},
    {"text": "How do I cancel my subscription and stop being charged?", "category": "billing"},
    # Technical
    {"text": "The app crashes every time I try to upload a file", "category": "technical"},
    {"text": "I'm getting a 500 error when accessing the dashboard", "category": "technical"},
    {"text": "The search feature returns no results even for exact matches", "category": "technical"},
    {"text": "My data export is stuck at 50% and won't complete", "category": "technical"},
    {"text": "The mobile app won't sync with the desktop version", "category": "technical"},
    {"text": "API rate limiting is too aggressive for our use case", "category": "technical"},
    # Account
    {"text": "I forgot my password and the reset email never arrives", "category": "account"},
    {"text": "How do I enable two-factor authentication on my account?", "category": "account"},
    {"text": "I need to transfer ownership of our team account", "category": "account"},
    {"text": "Can I merge my two accounts into one?", "category": "account"},
    {"text": "My account was locked after too many login attempts", "category": "account"},
    {"text": "I want to delete my account and all associated data", "category": "account"},
    # Feature request
    {"text": "It would be great to have dark mode in the app", "category": "feature_request"},
    {"text": "Can you add support for exporting to CSV format?", "category": "feature_request"},
    {"text": "We need bulk edit functionality for managing large datasets", "category": "feature_request"},
    {"text": "Please add keyboard shortcuts for common actions", "category": "feature_request"},
    {"text": "Integration with Slack would improve our workflow significantly", "category": "feature_request"},
    {"text": "A mobile widget for quick access would be very useful", "category": "feature_request"},
]

# Convert to DSPy Examples
examples = [dspy.Example(text=d["text"], category=d["category"]).with_inputs("text")
            for d in raw_data]

# Split into train / dev / test
random.seed(42)
random.shuffle(examples)
trainset = examples[:12]
devset = examples[12:18]
testset = examples[18:]

print(f"Train: {len(trainset)}, Dev: {len(devset)}, Test: {len(testset)}")
print(f"Categories: billing, technical, account, feature_request")
print(f"\nSample: '{trainset[0].text}' → {trainset[0].category}")

## Step 3 — Define Classifier with DSPy Signatures

A **Signature** declares what goes in and what comes out.
A **Module** wraps a signature with a prompting strategy.

In [ ]:
# Method 1: Inline signature with dspy.Predict
basic_classifier = dspy.Predict(
    "text -> category: str"  # inline signature
)

# Test it
pred = basic_classifier(text="I was charged twice this month")
print(f"Basic classifier: '{pred.category}'")

In [ ]:
# Method 2: Class-based signature (more control)
class TicketClassifier(dspy.Signature):
    """Classify a customer support ticket into exactly one category.
    Categories: billing, technical, account, feature_request."""
    
    text: str = dspy.InputField(desc="The customer support ticket text")
    category: str = dspy.OutputField(
        desc="One of: billing, technical, account, feature_request"
    )

# Wrap with ChainOfThought for better reasoning
cot_classifier = dspy.ChainOfThought(TicketClassifier)

pred = cot_classifier(text="The app keeps crashing on startup")
print(f"CoT classifier: '{pred.category}'")
print(f"Reasoning: {pred.reasoning[:200]}")

## Step 4 — Build a Classifier Module

A `dspy.Module` is like a PyTorch `nn.Module` —
it contains sub-modules and defines forward logic.

In [ ]:
class SupportClassifier(dspy.Module):
    """Multi-step classifier: classify then verify."""
    
    def __init__(self):
        super().__init__()
        self.classify = dspy.ChainOfThought(TicketClassifier)
    
    def forward(self, text):
        prediction = self.classify(text=text)
        # Normalize the category
        category = prediction.category.strip().lower().replace(" ", "_")
        valid = {"billing", "technical", "account", "feature_request"}
        if category not in valid:
            # Fallback: pick closest match
            for v in valid:
                if v in category or category in v:
                    category = v
                    break
            else:
                category = "technical"  # default fallback
        prediction.category = category
        return prediction

classifier = SupportClassifier()
pred = classifier(text="My invoice shows the wrong amount")
print(f"Module output: '{pred.category}'")

## Step 5 — Define Metric & Evaluate Baseline

In [ ]:
# Metric: exact match on category
def classification_accuracy(example, prediction, trace=None):
    """Return 1.0 if predicted category matches expected."""
    predicted = prediction.category.strip().lower().replace(" ", "_")
    expected = example.category.strip().lower().replace(" ", "_")
    return float(predicted == expected)

# Evaluate baseline (no optimization)
def evaluate_classifier(clf, dataset, name=""):
    correct = 0
    total = len(dataset)
    details = []
    
    for ex in dataset:
        pred = clf(text=ex.text)
        score = classification_accuracy(ex, pred)
        correct += score
        details.append({
            "text": ex.text[:50],
            "expected": ex.category,
            "predicted": pred.category,
            "correct": bool(score),
        })
    
    accuracy = correct / total
    print(f"\n{name} Accuracy: {accuracy:.0%} ({int(correct)}/{total})")
    for d in details:
        icon = "✅" if d["correct"] else "❌"
        print(f"  {icon} '{d['text']}...' → {d['predicted']} (expected: {d['expected']})")
    return accuracy

# Baseline evaluation
baseline_acc = evaluate_classifier(classifier, devset, "Baseline")

## Step 6 — Optimize with DSPy

`LabeledFewShot` selects the best few-shot examples
from your training set to include in the prompt.

In [ ]:
# Optimize with LabeledFewShot
optimizer = dspy.LabeledFewShot(k=4)  # include 4 examples in prompt

optimized_classifier = optimizer.compile(
    student=SupportClassifier(),
    trainset=trainset,
)

print("Optimization complete!")
print(f"Optimizer: LabeledFewShot (k=4)")
print(f"Training examples used: {len(trainset)}")

In [ ]:
# Evaluate optimized classifier
optimized_acc = evaluate_classifier(optimized_classifier, devset, "Optimized")

In [ ]:
# Compare on test set
print("\n" + "="*60)
print("FINAL COMPARISON ON TEST SET")
print("="*60)

baseline_test = evaluate_classifier(classifier, testset, "Baseline (test)")
optimized_test = evaluate_classifier(optimized_classifier, testset, "Optimized (test)")

print(f"\n{'='*60}")
print(f"Baseline:  {baseline_test:.0%}")
print(f"Optimized: {optimized_test:.0%}")
improvement = optimized_test - baseline_test
print(f"Improvement: {improvement:+.0%}")

## Step 7 — Inspect the Optimized Prompt

In [ ]:
# See what examples the optimizer selected
print("Optimized prompt includes these few-shot examples:")
print("="*60)

# Inspect the last LM call to see the full prompt
pred = optimized_classifier(text="I need help with my invoice")
dspy.inspect_history(n=1)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **dspy.LM** | `dspy.LM('ollama_chat/model')` — LLM wrapper |
| **dspy.configure** | Set the default LM globally |
| **Signature** | Declare inputs/outputs (`text -> category`) |
| **dspy.Predict** | Basic module that calls LM with signature |
| **dspy.ChainOfThought** | Adds reasoning before output |
| **dspy.Module** | Composable pipeline (like `nn.Module`) |
| **dspy.Example** | Training data format with `.with_inputs()` |
| **LabeledFewShot** | Optimizer that selects best few-shot demos |

## DSPy Flow
```
1. Define Signature          →  text -> category
2. Wrap in Module            →  ChainOfThought(TicketClassifier)
3. Prepare Examples          →  dspy.Example(text=..., category=...)
4. Define Metric             →  exact_match on category
5. Optimize                  →  LabeledFewShot.compile(student, trainset)
6. Evaluate                  →  compare baseline vs optimized
```

## Extending This Notebook
- Try `dspy.BootstrapFewShot` which generates synthetic examples
- Try `dspy.MIPROv2` for instruction + demo optimization
- Add confidence scoring to the classifier
- Test with more categories and edge cases
- Save/load optimized models with `classifier.save()` / `classifier.load()`